# 18 · Production smoke: Right column names and both-sides counts

Tests 4-5 of `docs/plans/prod_smoke_tests.md`. The Right side is read through the Databricks SQL warehouse with `dtrack_local.WarehouseTarget` (no Databricks Connect). It checks that every `col_map` right name and the Right `date_col` are spelled **exactly** like `DESCRIBE QUERY` reports them, and compares the per-day counts of both sides over the same window.

Read-only on both sides: only `DESCRIBE QUERY`, `SELECT ... GROUP BY` on the warehouse and `GROUP BY` counts on the Left. Each pair is checked on its own; a failing pair is recorded and the loop moves on.

## How to approach

- **Prerequisites**: kernel = the repo `.venv`; `uv sync --extra dbx` (databricks-sdk) plus `--extra athena` / `--extra ldap` if a Left source needs them. `~/.databrickscfg` profile `DBX_PROFILE` and a SQL warehouse id in `WAREHOUSE_ID` (`DTRACK_DBX_WAREHOUSE_ID`). Left credentials in the project `.env`.
- **Run order**: top to bottom. The warehouse cell may open a browser login. Test 4 is one `DESCRIBE QUERY` per pair (fast; a stopped warehouse adds start-up time). Test 5 counts both sides, one `GROUP BY` each.
- **What to look at**: the `wrong` table in test 4 (configured name vs. the correct spelling and its type) and the `differ` table in test 5.
- **Failure modes**: preflight `ok=False` = table/query name, grants, or `date_col` not in the Right columns; `disposition` errors on downloads = switch `DISPOSITION` to `inline`; no common day in test 5 = `date_col` / `date_type` / `where` differ between the sides.
- Test 4 catches the known bug: a right name whose case differs from `DESCRIBE` makes a numeric column silently categorical. It is not fixed yet, so fix the spelling in `col_map`.

In [ ]:
import os
from pathlib import Path

CONFIG_PATH = os.environ.get("DTRACK_PROD_CONFIG", "C:/work/dtrack/c.json")   # your real pairs config
ENV_FILE = os.environ.get("DTRACK_ENV_FILE", "C:/work/dtrack/.env")             # project .env with <macro>_USR / _PWD / _DSN
PAIR = os.environ.get("DTRACK_NB_PAIR", "orders")                           # used when PAIRS = [PAIR]
PAIRS = None                     # None = every active (non-skip) pair; or ["orders", "cards"]
TO_DATE = os.environ.get("DTRACK_PROD_TO") or None      # None = yesterday; or "YYYY-MM-DD"
FROM_DATE = os.environ.get("DTRACK_PROD_FROM") or None  # None = TO_DATE - (DAYS - 1)
DAYS = int(os.environ.get("DTRACK_PROD_DAYS", "3"))     # window length in days
DBX_PROFILE = os.environ.get("DTRACK_DBX_PROFILE", "corp")             # ~/.databrickscfg profile
WAREHOUSE_ID = os.environ.get("DTRACK_DBX_WAREHOUSE_ID", "")           # SQL warehouse id (required)
DISPOSITION = os.environ.get("DTRACK_LOCAL_DISPOSITION", "external_links")   # or "inline" when presigned downloads are blocked
WORKDIR = Path.home() / ".local/plans/nbs/_work/18"     # WarehouseTarget work dir (nothing is pulled here)

In [ ]:
# Load the project .env first: load_config() only searches from the current
# directory, and this notebook does not run from the repo root.
import copy
import json
import traceback
from datetime import date, timedelta

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

from dtrack_left.cli.config import load_config

pd.set_option("display.max_colwidth", 120)
if Path(ENV_FILE).exists():
    load_dotenv(ENV_FILE, override=False)
else:
    print(f"warning: {ENV_FILE} not found; Left credentials must already be in the environment")

# The real config: connection profiles resolved, col_map files expanded, validated.
config = load_config(CONFIG_PATH)

# Which pairs: every active (non-skip) pair, or the PAIRS list.
active = [name for name, spec in config["pairs"].items() if not spec.get("skip")]
if PAIRS is None:
    names = active
else:
    unknown = [name for name in PAIRS if name not in config["pairs"]]
    skipped = [name for name in PAIRS if name in config["pairs"] and name not in active]
    if unknown:
        print("not in the config, ignored:", unknown)
    if skipped:
        print("marked skip in the config, ignored:", skipped)
    names = [name for name in PAIRS if name in active]

# The check window: the last DAYS days up to TO_DATE (default yesterday).
TO = TO_DATE or (date.today() - timedelta(days=1)).isoformat()
FROM = FROM_DATE or (date.fromisoformat(TO) - timedelta(days=DAYS - 1)).isoformat()


# The pair exactly as configured, with fromDate/toDate narrowed to the window
# (a deep copy, so the loaded config is never changed).
def windowed(name):
    spec = copy.deepcopy(config["pairs"][name])
    spec["fromDate"], spec["toDate"] = FROM, TO
    return spec


WORKDIR.mkdir(parents=True, exist_ok=True)
print(f"window {FROM}..{TO}; {len(names)} pair(s): {names}")

## Connect to the SQL warehouse

`workspace(profile)` builds the `WorkspaceClient`; `Warehouse(client, id, disposition=...)` runs statements through the Statement Execution API.

In [ ]:
# One WorkspaceClient from ~/.databrickscfg (external-browser profiles open a
# login tab here), and the SQL warehouse read through the Statement Execution API.
from dtrack_left.backends.databricks import DEFAULT_PROFILE, workspace
from dtrack_local import Warehouse

client = workspace(DBX_PROFILE or DEFAULT_PROFILE)
warehouse = Warehouse(client, WAREHOUSE_ID, disposition=DISPOSITION)

# A trivial read proves auth, the warehouse id and the result disposition.
print(warehouse.rows("SELECT current_user(), current_catalog(), current_date()"))

## Right preflight (DESCRIBE QUERY)

`WarehouseTarget(warehouse, pair["right"], work_dir).preflight()` runs `DESCRIBE QUERY SELECT * FROM <relation>` and checks that the Right `date_col` is there. It returns the column names exactly as Databricks spells them, and keeps their types in `target.column_types`.

In [ ]:
# One target and one DESCRIBE per pair.
from dtrack_local.pipeline import WarehouseTarget

TARGETS, COLUMNS = {}, {}
rows = []
for name in names:
    spec = windowed(name)
    try:
        target = WarehouseTarget(warehouse, spec["right"], WORKDIR / "right")
        TARGETS[name] = target
        health = target.preflight()
        if health.get("ok"):
            COLUMNS[name] = list(health["columns"])
        rows.append({"pair": name, "relation": target.relation()[:80], "ok": bool(health.get("ok")),
                     "columns": len(health.get("columns") or []), "error": health.get("error", "")})
    except Exception as exc:
        rows.append({"pair": name, "ok": False, "error": repr(exc)})
preflight_right = pd.DataFrame(rows)
display(preflight_right)

## Test 4 · col_map right names match DESCRIBE exactly

Every Right name the pair uses (`right.date_col` plus each `col_map` value) must appear in `DESCRIBE` with the same case. For each one that does not, the table shows the correct spelling (case-insensitive match) or `missing`, and the Databricks type of the correct column.

In [ ]:
# Compare each wanted Right name against the DESCRIBE set, exact case.
wrong_rows, rows = [], []
for name in names:
    if name not in COLUMNS:
        rows.append({"pair": name, "ok": False, "note": "no DESCRIBE (see preflight)"})
        continue
    spec = windowed(name)
    actual = set(COLUMNS[name])
    types = getattr(TARGETS[name], "column_types", {}) or {}
    wanted = [("right.date_col", spec["right"]["date_col"])]
    wanted += [(f"col_map[{left!r}]", right) for left, right in spec["col_map"].items()]
    bad = 0
    for where, configured in wanted:
        if configured in actual:
            continue
        bad += 1
        suggested = next((column for column in actual if column.lower() == configured.lower()), "missing")
        wrong_rows.append({"pair": name, "where": where, "configured": configured, "suggested": suggested,
                           "problem": "case" if suggested != "missing" else "missing",
                           "describe_type": types.get(suggested, "")})
    rows.append({"pair": name, "checked": len(wanted), "wrong": bad, "ok": bad == 0, "note": ""})
display(pd.DataFrame(rows))
wrong = pd.DataFrame(wrong_rows)
if wrong.empty:
    print("every right name matches DESCRIBE exactly")
else:
    display(wrong)

In [ ]:
# The corrected col_map per pair, ready to paste (only case fixes are applied;
# "missing" names are left as configured and need a manual decision).
for name, group in (wrong.groupby("pair") if not wrong.empty else []):
    fixes = {row.configured: row.suggested for row in group.itertuples() if row.suggested != "missing"}
    if not fixes:
        continue
    spec = windowed(name)
    fixed = {left: fixes.get(right, right) for left, right in spec["col_map"].items()}
    print(f"--- {name}")
    if spec["right"]["date_col"] in fixes:
        print(f'right.date_col: "{fixes[spec["right"]["date_col"]]}"')
    print(json.dumps(fixed, indent=2, ensure_ascii=False))

## Test 5 · Both sides count the same window

The Left `partition_counts` and the Right `partition_counts` (one `GROUP BY` on the warehouse) for `FROM..TO`. At least one day must be present on both sides; days whose counts differ are **listed, not judged**: real data may legitimately differ.

In [ ]:
# Count both sides per pair; collect the per-day comparison for the tables below.
from dtrack_left.backends.sources import source_from_config

PER_DAY = {}
rows = []
for name in names:
    spec = windowed(name)
    try:
        left = source_from_config(spec["left"]).partition_counts(FROM, TO)
        if name not in TARGETS:
            raise RuntimeError("no Right target (see preflight)")
        right = TARGETS[name].partition_counts(FROM, TO)
        days = sorted(set(left) | set(right))
        frame = pd.DataFrame({"left": [left.get(d) for d in days], "right": [right.get(d) for d in days]},
                             index=pd.Index(days, name="day"))
        frame["diff"] = frame["left"].fillna(0).astype(int) - frame["right"].fillna(0).astype(int)
        PER_DAY[name] = frame
        common = set(left) & set(right)
        differ = [d for d in days if left.get(d) != right.get(d)]
        rows.append({"pair": name, "left_days": len(left), "right_days": len(right), "common_days": len(common),
                     "differ_days": len(differ), "left_rows": sum(left.values()), "right_rows": sum(right.values()),
                     "ok": bool(common), "error": ""})
    except Exception as exc:
        rows.append({"pair": name, "ok": False, "error": repr(exc)})
both_sides = pd.DataFrame(rows)
display(both_sides)

In [ ]:
# Only the days that differ, per pair (missing on one side shows as NaN).
for name, frame in PER_DAY.items():
    differ = frame[frame["left"] != frame["right"]]
    print(f"--- {name} {FROM}..{TO}: {len(differ)} differing day(s)")
    if not differ.empty:
        display(differ)

## What to check

- Preflight: every pair `ok=True`; the `relation` column shows what is actually queried.
- Test 4: the `wrong` table is empty. Otherwise copy the printed col_map; `describe_type` tells you whether a numeric column was about to be compared as categorical.
- Test 5: `common_days > 0` for every pair. Differing days are information: compare with what you expect from the load schedule.